# From Qiskit to Polypus

Self-contained reference guide for readers who already know how to build and run circuits with Qiskit and just need the direct mapping to Polypus, without going through a from-scratch introduction.

## Building a circuit

Same pattern, same idea of a qubit register, with method chaining in Polypus:

In [1]:
import polypus
from qiskit import QuantumCircuit

# Qiskit
qc_qiskit = QuantumCircuit(2)
qc_qiskit.h(0)
qc_qiskit.cx(0, 1)
qc_qiskit.measure_all()

# Polypus
qc_polypus = polypus.Circuit(2).h(0).cx(0, 1).measure_all()

## Gates: same names, watch the argument order

Gates without an angle are called exactly the same, with the same arguments in the same order:

| Gate | Qiskit and Polypus |
|---|---|
| Hadamard | `qc.h(q)` |
| Pauli X / Y / Z | `qc.x(q)`, `qc.y(q)`, `qc.z(q)` |
| S / Sdg | `qc.s(q)`, `qc.sdg(q)` |
| T / Tdg | `qc.t(q)`, `qc.tdg(q)` |
| CNOT | `qc.cx(control, target)` |
| CZ | `qc.cz(control, target)` |
| SWAP | `qc.swap(q0, q1)` |
| Measurement | `qc.measure(q, c)`, `qc.measure_all()` |

Gates with an angle have the same name, but **the argument order is reversed**: Qiskit puts the angle first, Polypus puts it last.

| Gate | Qiskit | Polypus |
|---|---|---|
| X / Y / Z rotation | `qc.rx(theta, q)` | `qc.rx(q, theta)` |
| Controlled phase | `qc.cp(theta, control, target)` | `qc.cp(control, target, theta)` |
| ZZ / XX | `qc.rzz(theta, q0, q1)` | `qc.rzz(q0, q1, theta)` |
| Generic U | `qc.u(theta, phi, lam, q)` | `qc.u(q, theta, phi, lam)` |

Copying and pasting a Qiskit call with an angle as is fails loudly, not silently: Polypus expects an integer in the qubit position, so an angle there raises an immediate `TypeError`, not a silently wrong result.

In [2]:
import math

try:
    polypus.Circuit(1).rx(math.pi / 3, 0)  # Qiskit's order, by mistake
except TypeError as e:
    print(e)

'float' object cannot be interpreted as an integer


## Running

Qiskit no longer has `execute()`, since version 1.0 it runs directly against a backend. Qiskit's `AerSimulator` corresponds to Polypus's `infrastructure="local"`:

In [3]:
# Qiskit
from qiskit_aer import AerSimulator

backend = AerSimulator()
qiskit_counts = (
    backend.run(qc_qiskit, shots=1000, seed_simulator=42).result().get_counts()
)

# Polypus
result = polypus.run_quantum_circuit(
    qc_polypus, shots=1000, infrastructure="local", seed=42
)
polypus_counts = result.counts[
    0
]  # counts[0]: a single run, list-vs-dict explained below

print(qiskit_counts)
print(polypus_counts)

{'00': 510, '11': 490}
{'00': 510, '11': 490}


Both results come out identical with the same seed: Polypus's `"aer"` backend, the one `infrastructure="local"` uses by default, is the same `AerSimulator` from Qiskit underneath. The real difference is in the shape of the result: Qiskit's `get_counts()` returns the dictionary directly, Polypus's `result.counts` wraps it in a list, with one element per parallel run requested (`n_qpus`); with a single run, as here, it is a list with one element. `result.merged_counts` gives the equivalent of Qiskit's single dictionary directly, regardless of `n_qpus`: the key-by-key sum of every element in the list, so `result.merged_counts == result.counts[0]` whenever there is only one.

## Without rewriting anything: Polypus runs a Qiskit `QuantumCircuit` as is

`run_quantum_circuit` accepts a Qiskit `QuantumCircuit` directly, with no conversion to `polypus.Circuit` beforehand. An already-written Qiskit circuit does not need to be touched to gain access to the rest of Polypus's infrastructures:

In [4]:
result = polypus.run_quantum_circuit(qc_qiskit, shots=1000, infrastructure="local")
print(result.counts[0])

{'00': 493, '11': 507}


## Circuits as OpenQASM 2.0 text

A circuit also travels as OpenQASM 2.0 text, both ways: `to_qasm2()` exports a Polypus circuit, and `polypus.Circuit.from_qasm2()` reads text back into one, re-exporting it byte-identically. Qiskit's own `qasm2.dumps()`/`qasm2.loads()` read and write the same format, so a circuit built in either library reaches the other as plain text, with no custom converter needed:

In [5]:
from qiskit import qasm2

qiskit_qasm = qasm2.dumps(qc_qiskit)
imported_from_qiskit = polypus.Circuit.from_qasm2(qiskit_qasm)

roundtrip_qasm = qc_polypus.to_qasm2()
imported_by_qiskit = qasm2.loads(roundtrip_qasm)

print(imported_from_qiskit.to_qasm2())
print(type(imported_by_qiskit))

OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0],q[1];
barrier q;
measure q -> c;

<class 'qiskit.circuit.quantumcircuit.QuantumCircuit'>


One gotcha on the way back into Qiskit: `swap`, `cp`, `rzz`, and `rxx` are gates Polypus exports that Qiskit's own parser does not recognise by default, an inconsistency between what Qiskit's exporter writes and what its parser accepts, not a Polypus limitation. Reading them back needs `qasm2.loads(text, custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS)`.

## Scaling beyond a local simulator

Where Qiskit uses a different backend, provider, or session depending on the hardware provider, Polypus changes a single argument, `infrastructure`: `"local"`, seen above, `"cunqa"` for simulated QPUs spread across SLURM, or `"qmio"` for real quantum hardware. The same circuit, unchanged, moves between the three.

## Training variational circuits

Qiskit does not ship its own optimizer for this: the usual approach is a manual loop with `scipy.optimize`, binding parameters and calling the backend on every evaluation:

In [6]:
# Qiskit + scipy
from qiskit.circuit import Parameter
from scipy.optimize import differential_evolution

theta = Parameter("theta")
training_qc = QuantumCircuit(1)
training_qc.h(0)
training_qc.rz(theta, 0)
training_qc.h(0)
training_qc.measure_all()


def cost(params):
    bound_qc = training_qc.assign_parameters({theta: params[0]})
    counts = backend.run(bound_qc, shots=500).result().get_counts()
    total = sum(counts.values())
    p0 = counts.get("0", 0) / total
    return -p0  # differential_evolution always minimizes


scipy_result = differential_evolution(
    cost, bounds=[(0, 2 * math.pi)], seed=3, maxiter=20, popsize=10
)
print(scipy_result.x, -scipy_result.fun)

[0.14694534] 1.0


`polypus.train()` replaces that whole manual loop, with no need to rewrite the circuit: it accepts the same Qiskit `QuantumCircuit` with `Parameter` from above as is, gets passed a function that receives a bitstring instead of a full counts dictionary, and Polypus takes care of binding, running, and averaging internally:

In [7]:
# Polypus
def bitstring_cost(bitstring):
    return 1.0 if bitstring == "0" else 0.0


polypus_result = polypus.train(
    training_qc,
    polypus.DE(generations=20, population_size=10, seed=3),
    shots=500,
    n_qpus=1,
    dimensions=1,
    expectation_function=bitstring_cost,
    infrastructure="local",
    nodes=1,
    cores_per_qpu=2,
    id="qiskit-to-polypus",
)
print(polypus_result.best_params, polypus_result.best_fitness)

[6.267204493647542] 1.0


The two angles found do not match as numbers, but they are equivalent by periodicity: both give `P(0) = 1.0`. One sign detail when migrating: scipy's `differential_evolution` always minimizes, hence the negative sign in `cost`; Polypus's `DE` always maximizes, so `bitstring_cost` does not need to flip anything. Polypus's other optimizers, `PSO` and `QNG`, share the same call to `train()`, changing only the object passed as `method`.

## Summary

Most already-built Qiskit code runs against Polypus without rewriting it: same gate names, with the argument order reversed on the ones that take an angle, and a full `QuantumCircuit` is accepted as is in both `run_quantum_circuit` and `train()`. What actually changes is how scaling works, `infrastructure` instead of backends and providers, and training, `train()` instead of a manual loop with `scipy`.